# Beam-Harvest Pilot — timed, resumable, max-parallel (A100 80 GB)

**Goal.** Run the trained PPO policy (`610model`) + beam search over the **first ~200**
presentations of `data/greedy_all.txt`, to (a) verify the full pipeline works on this A100
and (b) measure wall-clock so we can extrapolate the cost of the full 17,635-presentation run.

**Why batched.** The policy is tiny (~140k params), so one B=16,384 beam search uses only
~7–8 GB of the 80 GB A100 and does **not** saturate it. We batch many presentations per
`network.apply` via `jax.vmap` (a "wave") to fill the GPU, targeting **≤60 GB (20 GB buffer)**.

**Correctness.** The wave engine copies `beam/beam_search.py`'s `beam_step` **verbatim** (plus
two documented vmap-safety fixes). A **guard cell** runs the released `beam_search.py` as an
oracle on 3 presentations and asserts our engine produces byte-identical paths *before* any
result is trusted. The released code is the source of truth; we only batch it for speed.

**Resumable.** One JSON line is appended (and fsync'd) to a JSONL on Google Drive per finished
presentation, so a disconnect loses at most the in-flight wave; rerun skips completed indices.

> Paper config (Appendix I): B=16,384, T=150, α=0 (value head unused), Gumbel exploration,
> stop on first beam to reach the trivial presentation. The Gumbel **temperature** is the one
> value the paper does not pin; the primary pilot pass is deterministic (seed 0, temp 0).

In [ ]:
# Cell must run FIRST: XLA flags only take effect before jax is imported.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # grow memory on demand; required for
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"  # honest memory_stats and wave sizing
print("XLA configured: no preallocation, autotune off.")

## 1. Repo setup

Clone `github.com/Avi161/ACSolverX` (branch `test/eda`), `cd` into it, and install the
CUDA JAX stack. Uses `subprocess` (not IPython magics) so the notebook is portable and every
cell is plain Python. All entry points must run from the repo root (`data/` and
`ppo_checkpoints/` are relative paths).

In [ ]:
import subprocess, sys
from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH = "test/eda"
print("Environment:", "Google Colab" if IN_COLAB else "local")

if IN_COLAB:
    REPO_DIR = Path("/content/ACSolverX")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cuda.txt"], check=True)
else:
    REPO_DIR = Path.cwd()
    while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "requirements.txt").exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
    print("Local repo:", REPO_DIR, "(skipping clone/install)")

print("cwd:", os.getcwd())
print("branch:", subprocess.run(["git", "branch", "--show-current"],
                                 capture_output=True, text=True).stdout.strip())

## 2. Mount Google Drive

`RUN_DIR` on Drive is the only durable output location (local `/content` is wiped on
disconnect). All results + metadata are written there.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(os.getcwd()) / "_pilot_local"   # local fallback

RUN_DIR = DRIVE_ROOT / "beam_harvest_pilot"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

## 3. Regenerate the env inputs

`data/greedy_all.txt` is not committed (it is deterministically regenerated from the committed
greedy CSV). `scripts/csv_to_initial_states.py` writes it + `greedy_all_index.csv` and
self-validates (gate G1).

In [ ]:
subprocess.run([sys.executable, "scripts/csv_to_initial_states.py"], check=True)
n_lines = sum(1 for _ in open("data/greedy_all.txt"))
assert n_lines == 17635, f"expected 17635 lines, got {n_lines}"
print("greedy_all.txt ready:", n_lines, "presentations")

## 4. Pilot configuration

Paper Appendix I config. `SEEDS=[0]` + `TEMP/TEMP_END` define the optional 5-seed slice;
the primary 200-pass is deterministic (seed 0). `WAVE_SIZE=None` auto-picks from beam width.

In [ ]:
PILOT_CONFIG = {
    "N_PILOT": 200,
    "B": 16384,
    "T": 150,
    "ALPHA": 0.0,
    "DATASET": "greedy_all",
    "TRAINING_DATASET": "AC19_extended",
    "CKPT": "610model",
    "SEEDS": [0],         # explore seeds for the 5-seed slice cell; seed 0 is always temp 0
    "TEMP": 1.0,          # Gumbel temperature for explore seeds (unspecified in paper -> default)
    "TEMP_END": 0.0,
    "WAVE_SIZE": None,    # None -> auto-calibrated from a measured probe (see section 9b)
    "RUN_GUARD": True,    # run the released-oracle correctness guard. After it PASSES once,
                          # set False + restart runtime for the timing run (clean GPU, auto-sizing).
}
print(PILOT_CONFIG)

## 5. Imports, environment, checkpoint

Build `ACS` with `max_steps_in_episode = T = 150` (the **search horizon**, not the training
NUM_STEPS=96 — beams die on truncation otherwise). Restore **params only** (we never use the
informational train_solved/solve_data columns, so we skip that two-stage dance).

In [ ]:
import time, json, math, ast
import csv as _csv
import numpy as np
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
jax.config.update("jax_default_matmul_precision", "float32")

sys.path.insert(0, "scripts")           # for canon
from envs.ac_s import ACS
from envs.utils import decode_action, replay_packed_path
from network import RelativeDualRingActorCritic
import canon

print("JAX devices:", jax.devices())
try:
    _bl = jax.devices()[0].memory_stats().get("bytes_limit", 0)
    print(f"device memory limit: {_bl/1e9:.1f} GB")
except Exception as e:
    print("memory_stats unavailable:", e)

B = PILOT_CONFIG["B"]; T = PILOT_CONFIG["T"]; L = 24; ALPHA = float(PILOT_CONFIG["ALPHA"])
env = ACS(n_gen=2, max_length=L, max_steps_in_episode=T, is_reward_sparse=False,
          initial_states_file=PILOT_CONFIG["DATASET"])
env_params = env.default_params
network = RelativeDualRingActorCritic(activation="gelu")
n_actions = env.num_actions          # 4608; used verbatim as in beam_search.py (do NOT change)
obs_shape = env.observation_space(env_params).shape
obs_len = obs_shape[0]
print("n_actions:", n_actions, "obs_len:", obs_len)

In [ ]:
rng = jax.random.PRNGKey(0)
rng, init_rng = jax.random.split(rng)
dummy_params = network.init(init_rng, jnp.zeros((1, *obs_shape)))

ckpt_abs = os.path.join(os.getcwd(), "ppo_checkpoints", PILOT_CONFIG["CKPT"])
mngr = ocp.CheckpointManager(ckpt_abs, item_names=("params", "solve_data", "config"))
step = mngr.latest_step()
assert step is not None, f"no checkpoint at {ckpt_abs}"
print("restoring checkpoint step", step)
restored = mngr.restore(step, args=ocp.args.Composite(params=ocp.args.StandardRestore(dummy_params)))
params = restored.params
print("params restored (solve_data/config skipped: search needs only params)")

## 6. Shared constants + join index

`hash_vec` uses the **same seed `0xACABDED`** as `beam_search.py` so state hashes (hence the
dedup/visited logic) are identical — required for the guard to match byte-for-byte.

In [ ]:
GLOBAL_VISIT_CAP = B * T
HASH_SENTINEL = jnp.iinfo(jnp.int32).max
hash_vec = jax.random.randint(jax.random.PRNGKey(0xACABDED), (obs_len,),
                              minval=-(2**31), maxval=2**31 - 1, dtype=jnp.int32)

INDEX = {}   # line_idx -> (r1, r2, greedy_solved, greedy_path_length)
with open("data/greedy_all_index.csv") as f:
    for row in _csv.DictReader(f):
        INDEX[int(row["line_idx"])] = (row["r1"], row["r2"],
                                       row["greedy_solved"] == "True",
                                       int(row["greedy_path_length"]))
print("index rows:", len(INDEX))

## 7. The batched beam engine

`make_engine(B)` builds a `jax.vmap`-over-the-wave version of `beam_search.py`'s `beam_step`.
The inner `beam_step_batched` is **identical** to the released `beam_step` except:
1. a per-element `temp_sched` argument (so a wave can mix seeds/temperatures), and
2. two vmap-safety rewrites flagged in the CLAUDE.md lesson — `is_first` built from a batched
   slice (no scalar constant) and the `keep` scatter target as `jnp.zeros_like(alive)`.

`run_wave` initialises one independent B-beam search per wave element, steps all of them
together for up to T steps, pulls the (small) per-element terminated/alive flags to host each
step (vmap cannot `break` per element), and snapshots the winning packed path the moment an
element first reaches the trivial presentation. `run_batch` chunks a list of elements into
constant-size waves (padding the last wave so the jitted function never re-compiles).

In [ ]:
def make_engine(B):
    """Return (run_wave, run_batch) for beam width B. Closes over network/params/env/etc."""
    GLOBAL_VISIT_CAP = B * T

    def beam_step_batched(states, alive, cum_log_prob, action_seqs, visited_sorted, temp_sched, rng, t):
        # === verbatim from beam/beam_search.py:beam_step, with (1) temp_sched arg and
        #     (2) two vmap-safety fixes marked below. Logic is otherwise unchanged. ===
        rng, noise_rng, step_rng = jax.random.split(rng, 3)
        pi, value = network.apply(params, states.x)
        log_probs = jax.nn.log_softmax(pi.logits, axis=-1)
        action_scores = log_probs + ALPHA * value[:, None]
        u = jax.random.uniform(noise_rng, action_scores.shape, minval=1e-6, maxval=1.0 - 1e-6)
        gumbel = -jnp.log(-jnp.log(u))
        action_scores = action_scores + temp_sched[t] * gumbel              # per-element temp
        action_scores = jnp.where(alive[:, None], action_scores, -jnp.inf)
        candidate_scores = cum_log_prob[:, None] + action_scores
        flat_scores = candidate_scores.reshape(-1)
        top_vals, top_idx = jax.lax.top_k(flat_scores, B)
        parent = top_idx // n_actions
        flat_action = top_idx % n_actions
        action_k1 = (flat_action // (4 * L)) + 1
        rem = flat_action % (4 * L)
        action_k2_tmp = rem // 4
        ij = rem % 4
        action_i = ij // 2
        action_j = ij % 2
        action_k2 = action_k2_tmp * (-1) ** action_j - action_j
        action_vec = jnp.stack([action_i, action_j, action_k1, action_k2], axis=-1)
        parent_states = jax.tree.map(lambda v: v[parent], states)
        step_rngs = jax.random.split(step_rng, B)
        _, new_states, _, dones, info = jax.vmap(
            env.step_env, in_axes=(0, 0, 0, None))(step_rngs, parent_states, action_vec, env_params)
        new_action_seqs = action_seqs[parent].at[:, t].set(flat_action)
        noop_mask = jnp.all(parent_states.x == new_states.x, axis=-1)
        parent_alive = alive[parent]
        terminated = info["terminated"] & parent_alive
        new_alive = parent_alive & (~dones) & (top_vals > -1e30) & (~noop_mask)
        new_cum_log_prob = jnp.where(new_alive, top_vals, -jnp.inf)
        hashes = jnp.sum(new_states.x.astype(jnp.int32) * hash_vec[None, :], axis=1)
        sort_order = jnp.lexsort(jnp.stack([-new_cum_log_prob, hashes]))
        sorted_hashes = hashes[sort_order]
        # --- vmap fix 1: build is_first from a batched slice (no jnp.array([True]) constant) ---
        prev_hashes = jnp.concatenate([sorted_hashes[:1], sorted_hashes[:-1]])
        is_first = (sorted_hashes != prev_hashes).at[0].set(True)
        # --- vmap fix 2: zeros_like(alive), not zeros(B), as the scatter target ---
        keep = jnp.zeros_like(alive).at[sort_order].set(is_first)
        new_alive = new_alive & keep
        new_cum_log_prob = jnp.where(new_alive, new_cum_log_prob, -jnp.inf)
        idx_in_vis = jnp.searchsorted(visited_sorted, hashes)
        idx_clamped = jnp.minimum(idx_in_vis, GLOBAL_VISIT_CAP - 1)
        already_visited = (visited_sorted[idx_clamped] == hashes)
        new_alive = new_alive & (~already_visited)
        new_cum_log_prob = jnp.where(new_alive, new_cum_log_prob, -jnp.inf)
        new_entries = jnp.where(new_alive, hashes, HASH_SENTINEL)
        merged = jnp.concatenate([visited_sorted, new_entries])
        new_visited_sorted = jnp.sort(merged)[:GLOBAL_VISIT_CAP]
        return (new_states, new_alive, new_cum_log_prob, new_action_seqs,
                terminated, new_visited_sorted, rng)

    step_vmapped = jax.jit(jax.vmap(
        beam_step_batched, in_axes=(0, 0, 0, 0, 0, 0, 0, None)))

    def _init_element(idx):
        key = jax.random.PRNGKey(0)
        _, init_state = env.reset_env(key, env_params, idx=jnp.int32(int(idx)),
                                      sample=jnp.bool_(False), probs=None)
        bstate = jax.tree.map(
            lambda v: jnp.broadcast_to(jnp.asarray(v)[None], (B,) + jnp.asarray(v).shape),
            init_state)
        alive0 = jnp.ones(B, dtype=jnp.bool_)
        clp0 = jnp.full((B,), -jnp.inf, dtype=jnp.float32).at[0].set(0.0)
        aseq0 = jnp.full((B, T), -1, dtype=jnp.int32)
        init_hash = jnp.sum(jnp.asarray(init_state.x, dtype=jnp.int32) * hash_vec)
        vis0 = jnp.full((GLOBAL_VISIT_CAP,), HASH_SENTINEL, dtype=jnp.int32).at[0].set(init_hash)
        vis0 = jnp.sort(vis0)
        return bstate, alive0, clp0, aseq0, vis0

    def run_wave(elements):
        # elements: list of (idx, seed, temp, temp_end); length is the (constant) wave size.
        W = len(elements)
        sl_, al_, cl_, aq_, vs_, tp_, rg_ = [], [], [], [], [], [], []
        for (idx, seed, temp, temp_end) in elements:
            s, a, c, q, v = _init_element(idx)
            sl_.append(s); al_.append(a); cl_.append(c); aq_.append(q); vs_.append(v)
            tp_.append(jnp.linspace(float(temp), float(temp_end), T))
            rg_.append(jax.random.PRNGKey(int(seed)))
        states = jax.tree.map(lambda *xs: jnp.stack(xs), *sl_)
        alive = jnp.stack(al_); clp = jnp.stack(cl_); aseq = jnp.stack(aq_)
        vis = jnp.stack(vs_); temp_sched = jnp.stack(tp_); rngs = jnp.stack(rg_)

        solved = [False] * W; solved_len = [-1] * W; packed = [None] * W
        done = [False] * W
        for t in range(T):
            states, alive, clp, aseq, terminated, vis, rngs = step_vmapped(
                states, alive, clp, aseq, vis, temp_sched, rngs, jnp.int32(t))
            term_any = np.asarray(terminated.any(axis=1))
            alive_any = np.asarray(alive.any(axis=1))
            for w in range(W):
                if done[w]:
                    continue
                if term_any[w]:
                    bidx = int(np.asarray(terminated[w]).argmax())
                    sl = t + 1
                    solved[w] = True; solved_len[w] = sl
                    packed[w] = np.asarray(aseq[w, bidx, :sl]).tolist()
                    done[w] = True
                elif not alive_any[w]:
                    done[w] = True
            if all(done):
                break
        out = []
        for w, (idx, seed, temp, temp_end) in enumerate(elements):
            out.append({"idx": int(idx), "seed": int(seed),
                        "solved": bool(solved[w]),
                        "path_length": int(solved_len[w]) if solved[w] else -1,
                        "packed_path": packed[w] if solved[w] else []})
        return out

    def run_batch(elements, wave_size):
        results = []
        i = 0
        while i < len(elements):
            chunk = elements[i:i + wave_size]
            pad = wave_size - len(chunk)
            padded = chunk + [chunk[-1]] * pad if pad else chunk
            wave_out = run_wave(padded)
            results.extend(wave_out[:len(chunk)])   # drop padded duplicates
            i += wave_size
        return results

    return run_wave, run_batch

In [ ]:
RUN_WAVE, RUN_BATCH = make_engine(B)
print("engine built for B =", B)

## 8. Correctness guard (must pass before trusting the engine)

Run the **released** `beam/beam_search.py` on the first 3 presentations at temp 0, then run our
batched engine on the same 3 at wave sizes 1 and 3, and assert identical
`(solved, path_length, packed_path)`. At temp 0 the search is deterministic and RNG-independent
(the env ignores its key, Gumbel noise ×0), so a match proves the vmap reproduction is exact.
**If this fails, stop** — do not run the pilot; rerun the released `beam_search.py` directly.

> **Memory note.** The guard caches two compiled engine sizes and spins up a separate
> B=16,384 oracle process, leaving tens of GB retained on the GPU — that eats the headroom the
> timing run needs (it caused an OOM at wave 6). So run the guard **once** to verify, then set
> `RUN_GUARD=False`, **Runtime -> Restart session**, and re-run: the timing pass then starts on
> a clean GPU and the next cell auto-sizes the wave.

In [ ]:
if not PILOT_CONFIG.get("RUN_GUARD", True):
    print("RUN_GUARD=False -> guard skipped (already verified). "
          "Timing run starts on a clean GPU and auto-sizes the wave.")
else:
    GUARD_IDX = [0, 1, 2]
    guard_csv = str(RUN_DIR / "_guard.csv")
    cmd = [sys.executable, "beam/beam_search.py",
           "--ckpt_path", PILOT_CONFIG["CKPT"],
           "--dataset", PILOT_CONFIG["DATASET"],
           "--training_dataset", PILOT_CONFIG["TRAINING_DATASET"],
           "--start", "0", "--end", str(len(GUARD_IDX)),
           "--beam_width", str(B), "--max_steps", str(T),
           "--alpha", "0.0", "--temperature", "0.0", "--temp_end", "0.0",
           "--seed", "0", "--out_csv", guard_csv]
    print("oracle:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    import pandas as pd
    ref = pd.read_csv(guard_csv)
    def _parse_path(p):
        return ast.literal_eval(p) if isinstance(p, str) and p.strip().startswith("[") else []
    ref_map = {int(r.presentation_idx): (bool(r.solved), int(r.path_length), _parse_path(r.path))
               for r in ref.itertuples()}

    elements = [(i, 0, 0.0, 0.0) for i in GUARD_IDX]
    for ws in (1, len(GUARD_IDX)):
        got = RUN_BATCH(elements, ws)
        for g in got:
            rs, rl, rp = ref_map[g["idx"]]
            assert g["solved"] == rs and g["path_length"] == rl and g["packed_path"] == rp, (
                f"GUARD FAIL idx {g['idx']} wave={ws}: "
                f"ours {(g['solved'], g['path_length'])} vs released {(rs, rl)}")
        print(f"guard wave_size={ws}: OK ({len(got)} match released)")
    print("GUARD PASSED -- batched engine reproduces released beam_search.py byte-for-byte.")
    print(">>> Next: set RUN_GUARD=False, Runtime->Restart session, re-run for the timing pass. <<<")

## 9. Resume + JSONL persistence

One JSON object per presentation is appended to `pilot_results.jsonl` on Drive and fsync'd, so
a crash loses at most the in-flight wave. On rerun we read it, skip completed indices, and
refuse to mix a different B/T into the same file. `run_meta.json` is rewritten atomically.

In [ ]:
JSONL = RUN_DIR / "pilot_results.jsonl"
META = RUN_DIR / "run_meta.json"

def load_done():
    done = {}
    if JSONL.exists():
        with open(JSONL) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue   # tolerate a half-written final line from a crash
                done[rec["idx"]] = rec
    return done

def append_record(rec):
    with open(JSONL, "a") as f:
        f.write(json.dumps(rec) + "\n")
        f.flush(); os.fsync(f.fileno())

def write_meta(meta):
    tmp = str(META) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(meta, f, indent=2); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, META)

if META.exists():
    prev = json.load(open(META))
    assert prev.get("B") == B and prev.get("T") == T, (
        f"RUN_DIR already holds a run with B={prev.get('B')}, T={prev.get('T')}; "
        f"use a fresh RUN_DIR or matching config.")

done = load_done()
todo = [i for i in range(PILOT_CONFIG["N_PILOT"]) if i not in done]
print(f"resume: {len(done)} done, {len(todo)} to do")

## 9b. Auto-size the wave to the GPU (≤60 GB target)

Each presentation costs ~12–13 GB at B=16,384, so the wave size must be measured, not guessed.
On a **clean** GPU we probe with size-1 and size-2 waves (real searches on idx 0/1), read the
peak from `memory_stats`, derive the per-element cost, and pick the largest wave whose projected
peak stays under **60 GB** (20 GB buffer). If the GPU is **not clean** (the guard ran this
session → memory retained), we can't calibrate reliably, so we fall back to a conservative
default and tell you to restart. Set `PILOT_CONFIG["WAVE_SIZE"]` to override.

In [ ]:
FALLBACK_WAVE = {16384: 4, 8192: 8, 4096: 16, 2048: 32, 1024: 48, 512: 64}
TARGET_FRAC = 0.60   # use <=60% of VRAM (20 GB buffer on an 80 GB A100)

def _peak():
    try: return jax.devices()[0].memory_stats().get("peak_bytes_in_use", 0)
    except Exception: return 0
def _limit():
    try: return jax.devices()[0].memory_stats().get("bytes_limit", 80e9)
    except Exception: return 80e9

if PILOT_CONFIG["WAVE_SIZE"] is not None:
    wave_size = int(PILOT_CONFIG["WAVE_SIZE"])
    print(f"wave_size = {wave_size} (manual override)")
elif _peak() > 5e9:
    wave_size = FALLBACK_WAVE.get(B, 4)
    print(f"GPU NOT clean (peak {_peak()/1e9:.1f} GB -- guard/prior run retained memory). "
          f"Cannot calibrate; using conservative wave_size={wave_size}.\n"
          f"For auto-sizing + max throughput: set RUN_GUARD=False, Runtime->Restart, re-run.")
else:
    _ = RUN_WAVE([(0, 0, 0.0, 0.0)]); p1 = _peak()                       # 1-element probe
    _ = RUN_WAVE([(0, 0, 0.0, 0.0), (1, 0, 0.0, 0.0)]); p2 = _peak()     # 2-element probe
    marginal = max(p2 - p1, 1)
    fixed = max(p1 - marginal, 0)
    wave_size = max(1, int((TARGET_FRAC * _limit() - fixed) // marginal))
    print(f"calibration: fixed={fixed/1e9:.1f} GB, per-presentation={marginal/1e9:.2f} GB, "
          f"target={TARGET_FRAC*_limit()/1e9:.0f} GB -> wave_size={wave_size}")
    import gc as _gc; _gc.collect()

## 10. Main timed run (seed 0, deterministic)

Process `todo` in constant-size waves. The **first wave is warmup** (includes the one-time XLA
compile) and is excluded from timing. Each finished presentation is written + fsync'd
immediately. Wave wall-time is governed by its hardest member (a never-solving presentation
runs the full T=150), so per-presentation time is a fleet average.

In [ ]:
elements = [(i, 0, 0.0, 0.0) for i in todo]     # seed 0, temp 0 -> deterministic primary pass
wave_times = []                                  # (dt, n_real) for timed (non-warmup) waves
n_solved_run = sum(1 for r in done.values() if r.get("beam_solved"))
write_meta({"config": PILOT_CONFIG, "B": B, "T": T, "wave_size": wave_size,
            "engine": "batched", "n_done": len(done), "n_solved": n_solved_run})

for wi in range(0, len(elements), wave_size):
    chunk = elements[wi:wi + wave_size]
    pad = wave_size - len(chunk)
    padded = chunk + [chunk[-1]] * pad if pad else chunk
    w0 = time.time()
    wave_out = RUN_WAVE(padded)
    dt = time.time() - w0
    real = wave_out[:len(chunk)]
    for g in real:
        idx = g["idx"]
        r1, r2, gsolved, gpl = INDEX[idx]
        append_record({
            "idx": idx, "r1": r1, "r2": r2,
            "greedy_solved": gsolved, "greedy_path_length": gpl,
            "beam_solved": g["solved"], "beam_path_length": g["path_length"],
            "packed_path": g["packed_path"],
            "winning_seed": 0 if g["solved"] else -1,
            "wall_time": dt / len(chunk), "engine": "batched", "beam_width": B})
        n_solved_run += int(g["solved"])
    is_warmup = (wi == 0)
    if not is_warmup:
        wave_times.append((dt, len(chunk)))
    print(f"wave {wi // wave_size}: {len(chunk)} pres in {dt:.1f}s "
          f"({'warmup' if is_warmup else 'timed'}); solved-so-far {n_solved_run}")
    write_meta({"config": PILOT_CONFIG, "B": B, "T": T, "wave_size": wave_size,
                "engine": "batched", "n_done": len(load_done()),
                "n_solved": n_solved_run, "wave_times": wave_times})
print("main pass complete.")

## 11. Timing + full-run extrapolation

In [ ]:
N_FULL = 17635
total_t = sum(dt for dt, _ in wave_times)
total_p = sum(n for _, n in wave_times)
done = load_done()
n_solved = sum(1 for r in done.values() if r.get("beam_solved"))
n_shorter = sum(1 for r in done.values()
                if r.get("beam_solved") and r.get("greedy_solved")
                and r["beam_path_length"] < r["greedy_path_length"])
n_newsolve = sum(1 for r in done.values()
                 if r.get("beam_solved") and not r.get("greedy_solved"))

print(f"solved {n_solved}/{len(done)} | beam shorter than greedy: {n_shorter} | "
      f"new solves (greedy failed): {n_newsolve}")
if total_p:
    per_pres = total_t / total_p
    print(f"timed {total_p} presentations in {total_t:.1f}s -> {per_pres:.3f}s/pres "
          f"(B={B}, seed 0, post-warmup)")
    print(f"  full {N_FULL} run, 1 seed  : {per_pres * N_FULL / 3600:.2f} h")
    print(f"  full {N_FULL} run, 5 seeds : {per_pres * N_FULL * 5 / 3600:.2f} h "
          f"(upper bound; explore seeds often stop earlier)")
else:
    print("not enough timed waves for an estimate (need >1 wave).")
try:
    st = jax.devices()[0].memory_stats()
    print(f"GPU peak {st.get('peak_bytes_in_use', 0)/1e9:.1f} GB / "
          f"{st.get('bytes_limit', 0)/1e9:.1f} GB -> "
          f"{'room to raise wave_size' if st.get('peak_bytes_in_use',0) < 55e9 else 'near target'}")
except Exception as e:
    print("memory_stats unavailable:", e)

## 12. (Optional) Width-ladder timing at B=1,024

The full-run strategy is a cheap B=1,024 pass to clear the easy majority, then full
B=16,384×5 only on the unsolved remainder. This cell times the cheap pass over the same 200 so
we can compare flat-B vs ladder. Skip if you only need the headline B=16,384 number.

In [ ]:
RUN_LADDER = False   # set True to measure the B=1,024 pass

if RUN_LADDER:
    B_lo = 1024
    RUN_WAVE_lo, _ = make_engine(B_lo)
    ws_lo = FALLBACK_WAVE[B_lo]
    els = [(i, 0, 0.0, 0.0) for i in range(PILOT_CONFIG["N_PILOT"])]
    ltimes = []; lo_solved = 0
    for wi in range(0, len(els), ws_lo):
        chunk = els[wi:wi + ws_lo]
        pad = ws_lo - len(chunk)
        padded = chunk + [chunk[-1]] * pad if pad else chunk
        t0 = time.time(); out = RUN_WAVE_lo(padded); dt = time.time() - t0
        lo_solved += sum(int(o["solved"]) for o in out[:len(chunk)])
        if wi > 0:
            ltimes.append((dt, len(chunk)))
    tp = sum(n for _, n in ltimes); tt = sum(d for d, _ in ltimes)
    p_lo = lo_solved / PILOT_CONFIG["N_PILOT"]
    t_lo = tt / tp if tp else float("nan")
    print(f"B={B_lo}: solved {lo_solved}/{PILOT_CONFIG['N_PILOT']} (p_lo={p_lo:.2f}), "
          f"{t_lo:.3f}s/pres")
    if total_p:
        ladder_h = (t_lo * N_FULL + (1 - p_lo) * N_FULL * per_pres * 5) / 3600
        print(f"ladder estimate (B=1024 over all, then B=16384x5 on unsolved): {ladder_h:.2f} h")
else:
    print("RUN_LADDER=False (skipped)")

## 13. Validate harvested paths

Replay every solved packed path in a fresh env (sized to the path length) and assert it reaches
the trivial presentation in exactly `beam_path_length` steps — the repo's correctness test.

In [ ]:
done = load_done()
solved_recs = [r for r in done.values() if r.get("beam_solved")]
maxpl = max((r["beam_path_length"] for r in solved_recs), default=1)
val_env = ACS(n_gen=2, max_length=L, max_steps_in_episode=max(maxpl, 1),
              is_reward_sparse=False, initial_states_file=PILOT_CONFIG["DATASET"])
fails = []
for r in solved_recs:
    term, nsteps, _ = replay_packed_path(val_env, r["idx"], r["packed_path"], max_length=L)
    if not (term and nsteps == r["beam_path_length"]):
        fails.append(r["idx"])
print(f"replay: checked {len(solved_recs)} solved paths, {len(fails)} failed: {fails[:10]}")
assert not fails, f"{len(fails)} beam paths failed replay validation"
print("OK: every beam path reaches the trivial presentation in its recorded length.")

## 14. Harvest-format sample (Phase-3 preview)

Replay one solved path step-by-step and canonicalise each state via `scripts/canon.py` to show
the per-step trivialisation in the same canonical `xXyY` form as the greedy CSV.

In [ ]:
done = load_done()
demo = next((r for r in done.values() if r.get("beam_solved")), None)
if demo is None:
    print("no solved path to demo")
else:
    key = jax.random.PRNGKey(0)
    de = ACS(n_gen=2, max_length=L, max_steps_in_episode=demo["beam_path_length"],
             is_reward_sparse=False, initial_states_file=PILOT_CONFIG["DATASET"])
    _, st = de.reset_env(key, de.default_params, idx=jnp.int32(demo["idx"]), sample=jnp.bool_(False))
    traj = [canon.canonical_pair_str(*canon.env_state_to_strs(np.asarray(st.x), L))]
    for mv in [decode_action(a, L) for a in demo["packed_path"]]:
        _, st, _, _, _ = de.step_env(key, st, jnp.asarray(mv, dtype=jnp.int32), de.default_params)
        traj.append(canon.canonical_pair_str(*canon.env_state_to_strs(np.asarray(st.x), L)))
    print(f"idx {demo['idx']}: beam_len {demo['beam_path_length']} vs greedy_len "
          f"{demo['greedy_path_length']}; canonical trajectory ({len(traj)} states):")
    for s in traj:
        print("   ", s[0], "|", s[1])

## 15. Decision gate

Read off: (1) **GUARD PASSED**; (2) solve rate over 200 + how often beam beats greedy length +
new solves; (3) `s/presentation` and the full-run hour estimates (1 seed / 5 seeds / ladder);
(4) GPU peak vs 60 GB (whether `wave_size` can grow). Use these to choose the full-run config
(flat B=16,384×5 vs width-ladder) before launching all 17,635.